---

Database storage

---

In [1]:
# autoload
%load_ext autoreload
%autoreload 2
from pgl import pglDatabase, pglSession, pglDataset

---

Display all sessions in database

---

In [ ]:
pglDatabase.print()

---

Load a behavioral session

---

In [ ]:
session = pglSession.loadSession(104)

In [ ]:
session.history()

---

print session

---

In [ ]:
session.print()

In [ ]:
session.mne.displayReport()


---

Get behavioral data

---

In [ ]:
trials = session.getBehaviorTable("memoryDescriptionTask")

In [ ]:
# print a row of the table
#print(trials)
print(trials.iloc[1])

In [ ]:
# compute response times relative to a specific segment
trials = session.getBehaviorTable(taskName="memoryDescriptionTask", responseReference="segment", segmentNum=1)

---

Create a dataset

---

In [ ]:
dataset = pglDataset(name="Digital Brain Memory Pilot")
dataset.addSession([102,103,106,107,100,101,96,97,98,99,104,105])

---

get the behavioral table

---

In [ ]:
trials = dataset.getBehaviorTable(taskName="memoryDescriptionTask")

In [ ]:
# print a trial
print(trials.iloc[1000])

---

Save the dataset

---

In [ ]:
dataset.save()

---

Print datasets

---

In [ ]:
pglDatabase.print(datasets=True)

---

Load a dataset

---

In [ ]:
dataset = pglDataset.load(6)
dataset.print()

In [ ]:
remotePostgres.connectionTransport = "ssh"
remotePostgres.sshHost = "localhost"
remotePostgres.sshPort = 22
remotePostgres.sshUser = "justin"
remotePostgres.databaseHost = "127.0.0.1"
remotePostgres.databasePort = remotePostgres.postgresPort

verified = remotePostgres.verifyDatabase()
print(f"SQL-over-SSH verification: {verified}")

In [1]:
import hashlib
import posixpath
from uuid import uuid4

from pgl.pglSSH import pglSSHFilesystem
from pgl.pglStorage import pglStorage

access = remotePostgres.getSSHAccessConfiguration()
filesystem = pglSSHFilesystem(access)
testRoot = f"/Users/justin/data/pgl/storage/.pgl-ssh-test-{uuid4()}"
created = False

try:
    if not filesystem.isdir("/Users/justin/data/pgl/storage"):
        raise RuntimeError("Expected storage directory is unavailable through SSH.")

    filesystem.mkdir(testRoot)
    created = True

    binaryPath = posixpath.join(testRoot, "payload.bin")
    content = b"PGL verified SSH write test\n"

    with filesystem.open(binaryPath, "xb") as target:
        written = target.write(content)

    if written != len(content):
        raise AssertionError("Binary write did not return the expected byte count.")

    expected = {"sha256": hashlib.sha256(content).hexdigest(), "size_bytes": len(content)}
    actual = pglStorage.hashFile(filesystem, binaryPath)

    if actual != expected:
        raise AssertionError("Remote file identity differs from the written bytes.")

    exclusiveRejected = False

    try:
        with filesystem.open(binaryPath, "xb") as target:
            target.write(b"This must not replace the original.")
    except OSError:
        exclusiveRejected = True

    if not exclusiveRejected:
        raise AssertionError("Exclusive creation did not reject an existing file.")

    if pglStorage.hashFile(filesystem, binaryPath) != expected:
        raise AssertionError("The exclusive-create test changed the original file.")

    textPath = posixpath.join(testRoot, "metadata.txt")

    with filesystem.open(textPath, "xt", encoding="utf-8") as target:
        target.write("PGL Unicode test: α\n")

    with filesystem.open(textPath, "rt", encoding="utf-8") as source:
        restoredText = source.read()

    if restoredText != "PGL Unicode test: α\n":
        raise AssertionError("Text-mode round trip failed.")

    print("Passed: known-host-verified SSH connection.")
    print("Passed: binary write counts and SHA-256 verification.")
    print("Passed: exclusive creation preserved the existing file.")
    print("Passed: UTF-8 text writing and reading.")

finally:
    try:
        if created:
            filesystem.rm(testRoot, recursive=True)
    finally:
        filesystem.close()

print("Temporary SSH test directory removed; no database records changed.")

ModuleNotFoundError: No module named 'pgl.pglSSH'